<a href="https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use **Logistic Regression**.

My target is binary:

- `1` = declining
- `0` = not declining

Logistic Regression is a good first model because it is simple, fast, and interpretable. It also gives a probability for each content item, so I can rank pages from highest to lowest decline risk.

I will compare this model against my Week-4 rule baseline using the same test rows and the same ranking metrics.

I do not use `trend_direction` or `trend_pct` as model features because they are used to create the target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

import numpy as np
import pandas as pd


cwd = Path.cwd()

possible_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent
]

REPO_ROOT = cwd

for root in possible_roots:
    if (root / "data/raw/content_refresh_anonymized.csv").exists():
        REPO_ROOT = root
        break

repo_data = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"
uploaded_data = Path("content_refresh_anonymized.csv")

if repo_data.exists():
    DATA_PATH = repo_data
elif uploaded_data.exists():
    DATA_PATH = uploaded_data
else:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv"
    )

df = pd.read_csv(DATA_PATH)

# Target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

FEATURES = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

# Leakage guard
FORBIDDEN = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

assert not set(FEATURES).intersection(FORBIDDEN)

print("Dataset shape:", df.shape)
print("Features:", FEATURES)
print("Decline base rate:", round(df["is_declining_label"].mean(), 3))

Dataset shape: (30000, 45)
Features: ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr']
Decline base rate: 0.542


## 2. Split design

I use a **grouped train/test split by client**.

Pages from the same client may behave similarly, so putting pages from one client in both training and testing could make the result look better than it really is.

I therefore keep entire clients together. The model trains on about 80% of the clients and is tested on the remaining clients.

The random seed is fixed at `42` so the result can be reproduced.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

X = df[FEATURES].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"]

# avg_position = 0 means no position data
# Treat it as missing instead of rank 0.
X["avg_position"] = X["avg_position"].replace(0, np.nan)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print(
    "Clients appearing in both:",
    len(train_clients.intersection(test_clients))
)

assert train_clients.isdisjoint(test_clients)

print("\nPASS: no client appears in both train and test.")

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Clients appearing in both: 0

PASS: no client appears in both train and test.


## 3. Train + compare vs my baseline

I train one Logistic Regression model.

The model and the Week-4 rule are both evaluated only on the same held-out test clients.

The main metrics are:

- **Precision@10** — how many of the first 10 recommendations are actually declining.
- **Precision@50** — how many of the first 50 recommendations are actually declining.
- **ROC-AUC** — an additional measure of how well each method separates declining from non-declining content.

I also show the base rate so the ranking results have context.

The model only counts as an improvement if it beats the simple baseline on the same data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score



test_df = df.iloc[test_idx].copy()

visible = test_df["impressions_90d"] >= 300

stale = (
    test_df["days_since_last_update"] >= 180
)

ctr_opportunity = (
    (test_df["impressions_90d"] >= 300) &
    (test_df["avg_position"] > 0) &
    (test_df["avg_position"] <= 20) &
    (test_df["ctr"] < 1.0)
)

has_action_signal = stale | ctr_opportunity

baseline_scores = np.where(
    visible & has_action_signal,
    test_df["impressions_90d"] *
    (
        1 +
        stale.astype(int) +
        ctr_opportunity.astype(int)
    ),
    0
)




model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]




def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)

    k = min(k, len(order))

    return labels[order[:k]].mean()


base_rate = y_test.mean()



comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],

    "base_rate": [
        base_rate,
        base_rate
    ],

    "precision@10": [
        precision_at_k(
            baseline_scores,
            y_test,
            10
        ),

        precision_at_k(
            model_scores,
            y_test,
            10
        )
    ],

    "precision@50": [
        precision_at_k(
            baseline_scores,
            y_test,
            50
        ),

        precision_at_k(
            model_scores,
            y_test,
            50
        )
    ],

    "roc_auc": [
        roc_auc_score(
            y_test,
            baseline_scores
        ),

        roc_auc_score(
            y_test,
            model_scores
        )
    ]
})

display(
    comparison.round(3)
)

,method,base_rate,precision@10,precision@50,roc_auc
0,Week-4 baseline,0.511,0.5,0.42,0.504
1,Logistic Regression,0.511,0.5,0.66,0.517


In [ ]:
baseline_p50 = comparison.loc[
    comparison["method"] == "Week-4 baseline",
    "precision@50"
].iloc[0]

model_p50 = comparison.loc[
    comparison["method"] == "Logistic Regression",
    "precision@50"
].iloc[0]

print("Model vs baseline:")

if model_p50 > baseline_p50:
    print(
        "Logistic Regression beats the baseline at Precision@50."
    )

elif model_p50 < baseline_p50:
    print(
        "The Week-4 baseline beats Logistic Regression at Precision@50."
    )

else:
    print(
        "The model and baseline tie at Precision@50."
    )

Model vs baseline:
Logistic Regression beats the baseline at Precision@50.


## 4. Errors and interpretation

I do not judge the model only from its overall score.

I inspect:

1. false positives and false negatives,
2. which content types have the highest measured error rate,
3. the features Logistic Regression relies on most,
4. three confident predictions that were wrong.

The feature coefficients are interpreted as directional relationships with the prediction, not causal effects.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix


y_pred = (
    model_scores >= 0.5
).astype(int)



cm = confusion_matrix(
    y_test,
    y_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual not declining",
        "Actual declining"
    ],
    columns=[
        "Predicted not declining",
        "Predicted declining"
    ]
)

print("Confusion matrix:")
display(cm_df)




error_df = test_df[
    [
        "content_id",
        "content_type",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr"
    ]
].copy()

error_df["actual"] = y_test.values
error_df["predicted"] = y_pred
error_df["probability"] = model_scores

error_df["wrong"] = (
    error_df["actual"] !=
    error_df["predicted"]
)



error_by_type = (
    error_df
    .groupby("content_type")
    .agg(
        n=("content_id", "size"),
        error_rate=("wrong", "mean")
    )
    .reset_index()
    .sort_values(
        "error_rate",
        ascending=False
    )
)

error_by_type["error_rate"] = (
    error_by_type["error_rate"] * 100
).round(2)

print("Error rate by content type:")
display(error_by_type)




coefficients = model.named_steps[
    "model"
].coef_[0]

importance = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": coefficients
})

importance["absolute_importance"] = (
    importance["coefficient"].abs()
)

importance = importance.sort_values(
    "absolute_importance",
    ascending=False
)

print("Feature coefficients:")
display(importance)



error_df["confidence"] = np.where(
    error_df["predicted"] == 1,
    error_df["probability"],
    1 - error_df["probability"]
)

wrong_cases = (
    error_df[
        error_df["wrong"]
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .head(3)
)

print("Three confident wrong cases:")
display(wrong_cases)

Confusion matrix:


,Predicted not declining,Predicted declining
Actual not declining,504,2510
Actual declining,485,2664


Error rate by content type:


,content_type,n,error_rate
0,keyword article,6163,48.6


Feature coefficients:


,feature,coefficient,absolute_importance
3,ctr,-0.239154,0.239154
1,days_since_last_update,0.225600,0.225600
2,avg_position,-0.149109,0.149109
0,impressions_90d,-0.072404,0.072404


Three confident wrong cases:


,content_id,content_type,impressions_90d,days_since_last_update,avg_position,ctr,actual,predicted,probability,wrong,confidence
26844,content_8c19996aa890,keyword article,509252,20,2.5,0.15,1,0,0.122927,True,0.877073
21819,content_4c36c775b818,keyword article,463103,20,2.3,0.41,1,0,0.144336,True,0.855664
6653,content_5fe46e04994d,keyword article,517715,104,4.2,0.14,1,0,0.170695,True,0.829305


In [ ]:
false_positives = (
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
).sum()

false_negatives = (
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
).sum()

top3_features = (
    importance
    .head(3)["feature"]
    .tolist()
)

worst_type = error_by_type.iloc[0]

print(
    f"The model made {false_positives} false positives "
    f"and {false_negatives} false negatives."
)

print(
    f"The highest measured error rate was for "
    f"{worst_type['content_type']} "
    f"({worst_type['error_rate']:.2f}%)."
)

print(
    "The three strongest model features were:",
    ", ".join(top3_features)
)

print(
    "The three rows shown above are examples where "
    "the model was confident but still wrong."
)

The model made 2510 false positives and 485 false negatives.
The highest measured error rate was for keyword article (48.60%).
The three strongest model features were: ctr, days_since_last_update, avg_position
The three rows shown above are examples where the model was confident but still wrong.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.